# HIRECAR MarketWatch! — AI-Generated Workflow Blueprint

Uses the Anthropic Claude API to generate an interactive HTML workflow visual
covering all 16 departments, 82 AI bot workflows, gamification engine, and client milestone journey.

In [ ]:
import html
import os
import re
import time
import webbrowser
from datetime import datetime
from pathlib import Path

from anthropic import Anthropic
from IPython.display import HTML as DisplayHTML
from IPython.display import display

client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))


def save_html(html_content):
    os.makedirs("html_outputs", exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filepath = f"html_outputs/{timestamp}.html"
    with open(filepath, "w") as f:
        f.write(html_content)
    return filepath


def extract_html(text):
    pattern = r"```(?:html)?\s*(.*?)\s*```"
    matches = re.findall(pattern, text, re.DOTALL)
    return matches[0] if matches else None


def open_in_browser(filepath):
    abs_path = Path(filepath).resolve()
    webbrowser.open(f"file://{abs_path}")
    print(f"\U0001f310 Opened in browser: {filepath}")


def generate_html_with_claude(system_prompt, user_prompt):
    print("\U0001f680 Generating HTML...\n")

    full_response = ""
    start_time = time.time()
    display_id = display(DisplayHTML(""), display_id=True)

    with client.messages.stream(
        model="claude-sonnet-4-6",
        max_tokens=64000,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
    ) as stream:
        for text in stream.text_stream:
            full_response += text
            escaped_text = html.escape(full_response)
            display_html = f"""
            <div id="stream-container" style="border: 2px solid #667eea; border-radius: 8px; padding: 16px; background: #f8f9fa; max-height: 500px; overflow-y: auto;">
                <pre style="margin: 0; font-family: monospace; font-size: 12px; color: #2d2d2d; white-space: pre-wrap; word-wrap: break-word;">{escaped_text}</pre>
            </div>
            <script>
                requestAnimationFrame(() => {{
                    const container = document.getElementById('stream-container');
                    if (container) {{
                        container.scrollTop = container.scrollHeight;
                    }}
                }});
            </script>
            """
            display_id.update(DisplayHTML(display_html))

    elapsed = time.time() - start_time
    escaped_text = html.escape(full_response)
    final_html = f"""
    <div style="border: 2px solid #28a745; border-radius: 8px; padding: 16px; background: #f8f9fa; max-height: 500px; overflow-y: auto;">
        <pre style="margin: 0; font-family: monospace; font-size: 12px; color: #2d2d2d; white-space: pre-wrap; word-wrap: break-word;">{escaped_text}</pre>
    </div>
    """
    display_id.update(DisplayHTML(final_html))

    print(f"\n\u2705 Complete in {elapsed:.1f}s\n")

    html_content = extract_html(full_response)
    if html_content is None:
        print("\u274c Error: Could not extract HTML from response.")
        raise ValueError("Failed to extract HTML from Claude's response.")

    filepath = save_html(html_content)
    print(f"\U0001f4be HTML saved to: {filepath}")
    open_in_browser(filepath)

    return filepath

## System Prompt — HIRECAR Design System Expert

In [ ]:
SYSTEM_PROMPT = """
You are an expert UI/UX designer and front-end engineer specializing in interactive data visualization dashboards.
You work for HIRECAR — a premium auto operator intelligence platform.

DESIGN SYSTEM (MANDATORY — use these exact values):
- Background: #0d1117 (dark mode only)
- Ink (primary text): #111820
- Ink-2: #1e2530, Ink-3: #2d3540
- Card backgrounds: #1e2530
- CTA Gold: #c9920a (all primary actions, headings, highlights)
- Live Red: #c0392b (alerts, collision, emergency)
- Member Blue: #0f4c75 (membership, operator standards)
- Green: #00e676 (XP, positive metrics, success)
- Cyan: #00bcd4 (bot names, scoring, data)
- Purple: #7955d4 (scaling phase, elite tier)
- Orange: #e58827 (recovery, PIFR)
- Muted: #9aa3ad (secondary text)
- Border: #2d3540

FONTS:
- Brand/headings: 'Cormorant Garamond', serif (from Google Fonts)
- Body: 'Nunito Sans', sans-serif (from Google Fonts) 
- Mono/data: 'DM Mono', monospace (from Google Fonts)

RULES:
1. Output ONLY a single, complete, self-contained HTML file inside ```html``` code fences
2. All CSS must be inline in a <style> block — NO external CSS files
3. All JS must be inline in a <script> block — NO external JS files
4. Load Google Fonts via <link> tags
5. Make everything interactive: clickable cards, expandable sections, hover effects, smooth transitions
6. Use CSS Grid and Flexbox for layouts
7. The page must be responsive and look great on desktop (1440px+) and tablet (768px+)
8. Use subtle animations: hover scale, border-color transitions, fade-ins
9. Every data element must be accurate to the specification provided
10. The design should feel premium, dark, editorial — like a financial intelligence dashboard
"""

## User Prompt — Full Workflow Blueprint Specification

In [ ]:
USER_PROMPT = """
Create an interactive HIRECAR Workflow Blueprint dashboard. This is a comprehensive visualization of
our entire organizational system — 16 departments, 82 AI bot workflows, 5 client milestone phases,
and a gamification engine.

The page should have a sticky top bar with "HIRECAR MarketWatch!" branding and tab navigation.

═══════════════════════════════════════════════════════════════════════════════
TAB 1: CLIENT MILESTONE JOURNEY
═══════════════════════════════════════════════════════════════════════════════

Show a horizontal scrolling timeline of 5 phase cards connected by gold arrows.
Each card has a colored header matching the phase color, and expandable body showing:
- Entry criteria
- Active departments (as small pill badges)
- Scoring info
- Exit criteria  
- XP range badge (green mono font)

PHASE DATA:

Phase 1: INTAKE (color: #3466a4, icon: inbox)
- Entry: Lead captured via Apply to Qualify OR collision event reported
- Departments: CREDIT, COLLISION, INSURANCE, ROADSIDE
- Scoring: HBI baseline calculated, VDI initial scan, CRI if collision
- Exit: All intake data captured, baseline scores set, assigned to track
- XP: 0 - 500

Phase 2: RECOVERY (color: #c0392b, icon: wrench)
- Entry: Intake complete AND (HBI < 60 OR active collision claim OR collections)
- Departments: CREDIT, COLLISION, RECOVERY, COACHING, INTELLIGENCE
- Scoring: HBI improving, CRI actively tracked, disputes in progress
- Exit: No active disputes > 90 days, HBI >= 55, no new collections 60 days
- XP: 500 - 2,000

Phase 3: REBUILDING (color: #c9920a, icon: construction)
- Entry: Recovery exit criteria met AND assigned rebuilding track
- Departments: CREDIT, OPERATOR, RECOVERY, PLAYBOOKS, COACHING, INTELLIGENCE
- Scoring: HBI 55-75+, VDI being built, BRE in Cure Lane moving to Revenue
- Exit: HBI >= 70, VDI >= 80, BRE = Revenue Lane, all playbooks complete
- XP: 2,000 - 5,000

Phase 4: OPERATING (color: #00a651, icon: checkmark)
- Entry: BRE = Revenue Lane AND HBI >= 70 AND VDI >= 80
- Departments: OPERATOR, INSURANCE, FUNDING, MOBILITY, ROADSIDE, ENTERTAINMENT, INTELLIGENCE, PLAYBOOKS, COACHING, MEMBERSHIP, CHAUFFEUR, FLEET, CREDITCARD
- Scoring: All 6 scores active, composite score tracked, tier progression
- Exit: Composite >= 85, Fleet >= 5, Revenue >= $10K/mo, 6+ months
- XP: 5,000 - 15,000

Phase 5: SCALING (color: #7955d4, icon: rocket)
- Entry: Operating exit criteria met AND Elite tier application approved
- Departments: FUNDING, MOBILITY, ENTERTAINMENT, INTELLIGENCE, MEMBERSHIP, CHAUFFEUR, FLEET, CREDITCARD
- Scoring: All scores 85+, maximum XP accrual, Elite challenges/quests active
- Exit: N/A — continuous scaling, mentor status available
- XP: 15,000+

═══════════════════════════════════════════════════════════════════════════════
TAB 2: ALL 16 DEPARTMENTS
═══════════════════════════════════════════════════════════════════════════════

Show a 2-column grid of department cards. Each card has:
- Colored header bar with icon + department name
- Phase badges showing which milestone phases it's active in
- Bot count badge (green)
- Click to expand: reveals all bot workflows

Each bot workflow shows:
- Bot name (cyan, with robot emoji prefix)
- TRIGGER: label (gold) + condition text (muted)
- ACTION: label (green) + action text (muted)
- XP badge (green pill) + Score badge (cyan pill)

DEPARTMENT DATA (all 16 departments with every bot):

1. Credit Repair (HIRECREDIT) | icon: credit-card | color: #c9920a | phases: INTAKE, RECOVERY, REBUILDING
   - CreditScan Bot | trigger: Client onboarding OR collision event detected | action: Pull credit report, flag negatives, calculate HBI baseline | +100 XP | HBI
   - DisputeEngine Bot | trigger: HBI < 60 OR negative tradeline detected | action: Auto-generate dispute letters, file with bureaus, track 30/60/90 day cycles | +200 XP | HBI
   - CollectionsShield Bot | trigger: Collection account appears on report | action: Validate debt, send cease/desist, negotiate pay-for-delete | +150 XP | HBI
   - CreditCoach Bot | trigger: Client enters REBUILDING phase | action: Weekly credit tips, spending alerts, utilization monitoring | +50 XP | HBI
   - ApprovalPath Bot | trigger: HBI >= 70 AND client requests funding | action: Generate Approval Pathway Blueprint, map to funding products | +250 XP | FPI
   - TradeLine Bot | trigger: Dispute resolved OR new account opened | action: Update tradeline tracker, recalculate HBI, notify client | +75 XP | HBI

2. Collision + Claims | icon: zap | color: #c0392b | phases: INTAKE, RECOVERY
   - CollisionIntake Bot | trigger: New collision report filed OR police report uploaded | action: Parse collision data, identify all parties, create claims timeline | +150 XP | CRI
   - ClaimsCycle Bot | trigger: Collision intake complete | action: Track insurance claims across all parties, flag delays, auto-follow-up | +200 XP | CRI
   - PIFREngine Bot | trigger: Claims cycle > 30 days OR admin failure detected | action: Generate PIFR packet, compile evidence timeline, prepare dispute | +300 XP | CRI
   - CustodyLog Bot | trigger: Vehicle enters repair OR rental custody | action: Track vehicle custody chain, document handoffs, flag gaps | +100 XP | VDI
   - AdminFailure Bot | trigger: Insurance company misses SLA OR documentation error found | action: Flag admin failure, generate complaint letter, escalate to supervisor | +175 XP | CRI

3. Operator Standards | icon: clipboard | color: #0f4c75 | phases: REBUILDING, OPERATING
   - HBIScoring Bot | trigger: Weekly schedule OR manual trigger by client | action: Calculate HBI score (0-100), flag cure items, generate report | +100 XP | HBI
   - VDIAudit Bot | trigger: Document uploaded OR monthly VDI cycle | action: Audit vehicle documents, check expiration dates, score VDI (0-100) | +125 XP | VDI
   - BREEngine Bot | trigger: HBI + VDI scores updated | action: Run Business Readiness Engine, classify Revenue vs Cure Lane | +150 XP | BRE
   - VendorAudit Bot | trigger: New vendor added OR quarterly review cycle | action: Verify vendor credentials, insurance, compliance. Score vendor reliability | +100 XP | BRE
   - CureWindow Bot | trigger: BRE classifies client in Cure Lane | action: Set 30-day cure window, daily check-ins, remediation tasks | +200 XP | BRE
   - RevenueGate Bot | trigger: Cure window complete AND all items resolved | action: Clear for Revenue Lane, unlock premium services, notify account team | +300 XP | BRE

4. Recovery + PIFR | icon: refresh | color: #e58827 | phases: RECOVERY, REBUILDING
   - PIFRTriage Bot | trigger: New incident report OR collision claim filed | action: Assess incident severity, recommend PIFR tier (Core/Plus/Max/Shield) | +150 XP | CRI
   - RecoveryTimeline Bot | trigger: PIFR tier selected | action: Build full recovery timeline, set milestones, assign tasks | +200 XP | CRI
   - EvidenceCompiler Bot | trigger: New document uploaded for PIFR case | action: Organize evidence, check completeness, flag missing items | +100 XP | CRI
   - SettlementTracker Bot | trigger: Settlement offer received OR negotiation milestone | action: Track all settlement offers, compare to target, recommend accept/counter | +250 XP | FPI
   - RecoveryScore Bot | trigger: Monthly assessment OR case milestone reached | action: Calculate recovery progress %, update CRI score, adjust timeline | +75 XP | CRI

5. Auto Insurance | icon: shield | color: #00bcd4 | phases: INTAKE, OPERATING
   - PolicyScan Bot | trigger: Client onboarding OR policy renewal date approaching | action: Analyze current coverage, identify gaps, recommend upgrades | +125 XP | FPI
   - QuoteEngine Bot | trigger: Coverage gap identified OR client requests quote | action: Pull multi-carrier quotes, compare rates, present options | +150 XP | FPI
   - FleetCoverage Bot | trigger: New vehicle added to fleet OR annual fleet review | action: Ensure all fleet vehicles covered, batch policy updates | +100 XP | VDI
   - ClaimsAssist Bot | trigger: Insurance claim filed | action: Guide through claims process, track status, flag delays | +175 XP | CRI
   - BillingFog Bot | trigger: Payment missed OR billing discrepancy detected | action: Analyze billing, identify errors, dispute overcharges, prevent collections | +200 XP | FPI

6. Business Funding (SeedXchange) | icon: dollar | color: #00e676 | phases: OPERATING, SCALING
   - FundingReady Bot | trigger: Client enters OPERATING phase OR requests funding assessment | action: Run funding readiness check: HBI, BRE, revenue, time-in-business | +200 XP | FPI
   - SeedXchange Bot | trigger: Funding readiness score >= 70 | action: Submit to SeedXchange, 24hr qualification, match with capital sources | +300 XP | FPI
   - FleetExpansion Bot | trigger: Funding approved AND fleet growth plan submitted | action: Calculate fleet expansion ROI, recommend vehicle types, project revenue | +250 XP | MSI
   - CapitalDeploy Bot | trigger: Funds disbursed | action: Track capital deployment, monitor ROI, flag underperformance | +150 XP | FPI
   - RepayTracker Bot | trigger: Repayment schedule active | action: Monitor payments, send reminders, flag risk of default | +100 XP | FPI

7. Mobility + Rentals | icon: car | color: #0f4c75 | phases: OPERATING, SCALING
   - FleetMatch Bot | trigger: Client searches vehicles OR reservation request | action: Match to available fleet, check eligibility, present options with pricing | +100 XP | MSI
   - Reservation Bot | trigger: Vehicle selected by client | action: Process reservation, collect deposit, confirm availability, send confirmation | +150 XP | MSI
   - HandoffCoordinator Bot | trigger: Reservation date approaches (24hr) | action: Coordinate vehicle prep, confirm pickup details, send directions | +75 XP | MSI
   - FleetHealth Bot | trigger: Daily fleet scan OR vehicle return | action: Inspect vehicle condition, log mileage, schedule maintenance if needed | +100 XP | VDI
   - RentalRevenue Bot | trigger: Monthly cycle | action: Calculate rental revenue per vehicle, utilization rates, pricing optimization | +125 XP | MSI

8. Roadside + Emergency | icon: siren | color: #c0392b | phases: INTAKE, OPERATING
   - EmergencyDispatch Bot | trigger: Emergency request (roadside button OR call) | action: Locate client via GPS, dispatch nearest provider, ETA tracking | +200 XP | MSI
   - TowCoordinator Bot | trigger: Vehicle non-drivable confirmed | action: Arrange tow, coordinate with repair shop, update custody log | +150 XP | VDI
   - EVCharge Bot | trigger: EV battery low alert OR charge request | action: Locate nearest charger, reserve spot, provide routing | +75 XP | MSI
   - MaintenanceScheduler Bot | trigger: Mileage threshold OR service interval reached | action: Schedule oil change/tire/battery service, send reminders, confirm booking | +100 XP | VDI

9. Entertainment + Culture | icon: film | color: #7955d4 | phases: OPERATING, SCALING
   - ContentCurator Bot | trigger: New content published OR weekly content calendar | action: Curate city-specific entertainment, auto-tag, push to relevant clients | +50 XP | MSI
   - EventPromoter Bot | trigger: Event created OR 7 days before event | action: Generate event promotion, target eligible members, track RSVPs | +100 XP | MSI
   - CommunityEngagement Bot | trigger: Client completes 3+ interactions in week | action: Suggest community content, invite to events, award engagement XP | +75 XP | MSI
   - StorySpotlight Bot | trigger: Operator milestone achieved (tier up, badge earned) | action: Feature operator story, create content piece, share across channels | +150 XP | MSI

10. MW Intelligence | icon: chart | color: #00bcd4 | phases: RECOVERY, REBUILDING, OPERATING, SCALING
    - DataCollector Bot | trigger: New collision data available OR monthly data refresh | action: Ingest LA collision data (810K+ records), update MCI dataset | +100 XP | MSI
    - ReportGenerator Bot | trigger: Client requests report OR quarterly schedule | action: Generate custom intelligence report, market analysis, competitor data | +200 XP | MSI
    - HBIDistribution Bot | trigger: Monthly HBI recalculation complete | action: Generate HBI distribution chart, rank operator, identify trends | +150 XP | HBI
    - AlertEngine Bot | trigger: Market anomaly detected OR threshold breach | action: Push real-time alert to operators, recommend actions | +125 XP | MSI
    - ExportEngine Bot | trigger: Data export requested by client | action: Package data export (239K format), apply access controls, deliver | +75 XP | MSI

11. Playbooks + Guides | icon: book | color: #c9920a | phases: REBUILDING, OPERATING
    - PlaybookAssign Bot | trigger: Client enters new phase OR score drops below threshold | action: Assign relevant playbook, create checklist, set deadlines | +100 XP | BRE
    - ChecklistTracker Bot | trigger: Playbook assigned OR checklist item due | action: Track checklist progress, send reminders, verify completions | +75 XP | BRE
    - ExhibitReady Bot | trigger: Client approaching audit OR compliance review | action: Pre-audit check, compile required documents, generate exhibit packet | +200 XP | VDI
    - GuideRecommender Bot | trigger: Score improvement stalls OR client asks for help | action: Analyze gaps, recommend specific playbook sections, create action plan | +125 XP | BRE

12. CreditWithKen (Coaching) | icon: graduation | color: #c9920a | phases: RECOVERY, REBUILDING, OPERATING
    - SessionScheduler Bot | trigger: Client books coaching OR follow-up due | action: Schedule 1-on-1 session, send prep materials, set agenda | +100 XP | HBI
    - CoachPrep Bot | trigger: 24hr before scheduled session | action: Pull client data, generate coaching brief, highlight key focus areas | +75 XP | HBI
    - ActionPlan Bot | trigger: Post-session (coaching completed) | action: Generate action items from session, set deadlines, track follow-through | +150 XP | HBI
    - ProgressTracker Bot | trigger: Weekly check-in OR action item due date | action: Review progress on coaching goals, celebrate wins, flag at-risk items | +100 XP | HBI

13. First Class Membership | icon: star | color: #c9920a | phases: OPERATING, SCALING
    - QualificationEngine Bot | trigger: Apply to Qualify submitted | action: Run qualification matrix: HBI, BRE, revenue, fleet size, time-in-business | +200 XP | MSI
    - MemberOnboard Bot | trigger: Qualification approved | action: Create member profile, assign tier, activate benefits, send welcome kit | +300 XP | MSI
    - TierMonitor Bot | trigger: Monthly score update OR significant activity | action: Check tier eligibility, auto-upgrade/downgrade, notify member | +100 XP | MSI
    - BenefitsEngine Bot | trigger: Member accesses benefit OR new benefit available | action: Serve eligible benefits, track usage, recommend unused perks | +75 XP | MSI
    - RenewalBot | trigger: 30 days before membership renewal | action: Assess member value, generate renewal offer, process auto-renewal | +150 XP | MSI

14. Chauffeur + Private Driver | icon: hat | color: #2d3540 | phases: OPERATING, SCALING
    - DriverMatch Bot | trigger: Chauffeur/Private Driver booking request | action: Match available driver, check vehicle, confirm booking, send ETA | +150 XP | MSI
    - RouteOptimizer Bot | trigger: Booking confirmed | action: Optimize route, calculate fare, set pickup/dropoff coordinates | +75 XP | MSI
    - DriverRating Bot | trigger: Trip completed | action: Collect rating, update driver score, flag issues, reward excellence | +50 XP | MSI
    - FleetScheduler Bot | trigger: Daily morning schedule OR new booking added | action: Optimize driver-vehicle assignments, minimize deadheading, balance load | +100 XP | MSI

15. Fleet Management | icon: truck | color: #3d4a58 | phases: OPERATING, SCALING
    - VehicleOnboard Bot | trigger: New vehicle acquired | action: Register vehicle, set up VDI profile, schedule initial inspection, assign fleet ID | +200 XP | VDI
    - MaintenanceAI Bot | trigger: Mileage threshold OR time interval OR alert | action: Schedule maintenance, coordinate with shop, update service history | +125 XP | VDI
    - ComplianceWatch Bot | trigger: Registration/insurance/smog expiration approaching | action: Alert owner, auto-schedule renewal, track completion | +150 XP | VDI
    - DepreciationTracker Bot | trigger: Monthly valuation cycle | action: Calculate vehicle value, depreciation rate, recommend sell/keep | +100 XP | FPI
    - UtilizationBot | trigger: Weekly fleet review | action: Analyze per-vehicle utilization, revenue per mile, recommend rebalancing | +125 XP | MSI

16. HIRECAR Credit Card | icon: diamond | color: #c9920a | phases: OPERATING, SCALING
    - CardQualification Bot | trigger: Credit card application submitted | action: Check HBI score, credit history, membership status, approve/deny | +200 XP | FPI
    - CardOnboard Bot | trigger: Application approved | action: Issue virtual card, set limits, activate rewards program | +250 XP | FPI
    - SpendAnalytics Bot | trigger: Transaction posted OR weekly summary | action: Categorize spend, track rewards earned, identify optimization opportunities | +75 XP | FPI
    - RewardsEngine Bot | trigger: Qualifying transaction OR reward threshold reached | action: Award points/cashback, notify member, suggest redemption options | +50 XP | FPI
    - FraudWatch Bot | trigger: Unusual transaction pattern detected | action: Flag transaction, alert cardholder, temporary hold if high risk | +100 XP | FPI

═══════════════════════════════════════════════════════════════════════════════
TAB 3: GAMIFICATION ENGINE
═══════════════════════════════════════════════════════════════════════════════

Show in a 2-column grid:

Left panel — XP TIERS (4 cards in a row):
- Standard: 0 XP (color: #6b7280)
- Operator: 1,000 XP (color: #00bcd4)
- First Class: 5,000 XP (color: #c9920a)
- Elite: 15,000 XP (color: #7955d4)

Right panel — AI WORKFLOW PATTERN:
GREET -> GUIDE -> ACT -> REWARD -> NEXT
(show as connected step badges with green borders and gold arrows between)

Full-width panel — 6 SCORING SYSTEMS (3x2 grid):
- HBI: Health & Business Index (0-100) — Credit health + business viability
- VDI: Vehicle Documentation Index (0-100) — Fleet documentation completeness
- BRE: Business Readiness Engine (0-100) — Revenue vs Cure Lane classification
- CRI: Claims Recovery Index (0-100) — Active claims + recovery progress
- FPI: Financial Performance Index (0-100) — Revenue, funding, card activity
- MSI: Member Services Index (0-100) — Engagement, services, community
Each score card should have the acronym large in cyan, name, description, range, and an animated fill bar.

COMPOSITE SCORE FORMULA (highlight box):
Composite = (HBI x 0.25) + (VDI x 0.15) + (BRE x 0.20) + (CRI x 0.10) + (FPI x 0.15) + (MSI x 0.15)

Full-width panel — 9 ACHIEVEMENT BADGES (3x3 grid):
1. First Scan (search icon) — Complete first HBI scan — +100 XP
2. Document Pro (file icon) — VDI reaches 80+ — +250 XP
3. Revenue Ready (dollar icon) — BRE classifies Revenue Lane — +500 XP
4. Recovery Champion (trophy icon) — Complete a PIFR case successfully — +750 XP
5. Fleet Commander (car icon) — Register 3+ vehicles — +400 XP
6. Credit Hero (credit-card icon) — HBI improves 20+ points — +600 XP
7. Funded (money icon) — Receive first SeedXchange funding — +1,000 XP
8. First Class (star icon) — Achieve First Class Membership — +1,500 XP
9. Elite Operator (crown icon) — Reach Elite tier (15,000 XP) — +2,000 XP

═══════════════════════════════════════════════════════════════════════════════
TAB 4: BOT TRIGGER TABLE
═══════════════════════════════════════════════════════════════════════════════

A full searchable/filterable table of all 82 bot workflows:
- Search box at top (searches across all columns)
- Filter buttons by score type: All | HBI | VDI | BRE | CRI | FPI | MSI
- Table columns: Department | Bot Name | Trigger Condition | Action | XP | Score
- Styled with the design system colors
- Rows should highlight on hover
- Sortable columns (click header to sort)

═══════════════════════════════════════════════════════════════════════════════
TAB 5: FUNNEL CONVERGENCE MAP
═══════════════════════════════════════════════════════════════════════════════

Visual funnel diagram showing:
- Top: "ALL FRONT PAGE LINKS" box (219 total: 57 active, 160 placeholder, 3 external)
- Arrow down to 3 funnel type boxes side by side:
  - CONTENT FUNNEL (cyan border): Read/Info -> Desktop Mode -> Scroll engagement (~57 links)
  - SERVICE FUNNEL (gold border): Reserve/Purchase -> Slide 5 Purchase Carousel (~30 CTAs)
  - EXTERNAL HANDOFF (gray border): HIRECAR.LA exits (3 external)
- Arrow down to 6 product boxes: Credit Card, Membership, PIFR, Insurance, Chauffeur, MW Report
- Arrow down to terminal conversion box (large, gold border):
  - "APPLY TO QUALIFY — Terminal Conversion Point"
  - "Lead Capture -> Verify -> Thank You -> LIVE Access"

Also show the Remote Entry State Machine:
idle -> tronOverlay -> leadModal -> verify -> thankYou -> heroRevealed

═══════════════════════════════════════════════════════════════════════════════
STATS BAR (visible on all tabs)
═══════════════════════════════════════════════════════════════════════════════
Show at top of first tab: 8 stat cards in a row:
5 Phases | 16 Departments | 82 AI Bots | 6 Scoring Systems | 4 Tiers | 9 Badges | 219 Links | 3 Funnels

FOOTER:
"HIRECAR MarketWatch! — Workflow Blueprint v2.0 — Generated 2026-03-03 — Auto Operator Intelligence"
"""

## Generate the HTML Visual

In [ ]:
filepath = generate_html_with_claude(SYSTEM_PROMPT, USER_PROMPT)
print(f"\n\U0001f3af Output file: {filepath}")

## Preview the Generated HTML

In [ ]:
with open(filepath, 'r') as f:
    generated_html = f.read()

print(f"Generated HTML: {len(generated_html):,} characters")
display(DisplayHTML(f'<iframe srcdoc="{html.escape(generated_html)}" style="width:100%;height:800px;border:2px solid #c9920a;border-radius:8px;"></iframe>'))